# 📘 Lesson 13: Recurrent Neural Networks (RNN) from Scratch

**Step-by-Step Interactive Homework Notebook** with modular code execution and detailed explanations.


### 🔹 Select Device

**Purpose**: Import required libraries and frameworks (e.g. PyTorch, NumPy, Sklearn, Hugging Face).

- Sets up the execution environment, random seeds, and GPU/MPS device acceleration if available.


In [ ]:
import torch
import random
import torch.nn as nn
import matplotlib.pyplot as plt

from datasets import load_dataset
from torch.utils.data import Dataset, DataLoader

device = torch.device(
    "mps"
    if torch.backends.mps.is_available()
    else "cuda"
    if torch.cuda.is_available()
    else "cpu"
)
print(device)


### 🔹 Random Seed
Load TinyStories

**Purpose**: Data Ingestion and Exploration.

- Loads raw datasets into memory, inspects shape, distributions, and initial sample structures.


In [ ]:
torch.manual_seed(42)
random.seed(42)

dataset=load_dataset("roneneldan/TinyStories")

train_data=dataset["train"]
validation_data=dataset["validation"]

print(train_data[0])


### 🔹 Small Subset
Build Vocabulary

**Purpose**: Core functional execution step.

- Executes the defined transformation, evaluation, or helper utility.


In [ ]:
train_text = " ".join(train_data[:1000]["text"])
valid_text = " ".join(validation_data[:200]["text"])

all_text = train_text + valid_text

chars = sorted(list(set(all_text)))

stoi={c:i for i, c in enumerate(chars)}
itos={i:c for c, i in stoi.items()}

vocab_size=len(chars)
print(vocab_size)


### 🔹 Encode
Custom Dataset

**Purpose**: PyTorch Dataset & DataLoader Pipeline.

- Wraps tensors in iterable batches, handles multi-threaded worker loading, dynamic collation, and shuffling.


In [ ]:
train_encoded=torch.tensor([stoi[c] for c in train_text], dtype=torch.long)
valid_encoded=torch.tensor([stoi[c] for c in valid_text], dtype=torch.long)

class TinyStoriesDataset(Dataset):
    def __init__(self, data, block_size):
        self.data=data
        self.block_size=block_size
        
    def __len__(self):
        return len(self.data)-self.block_size
    
    def __getitem__(self, idx):
        x=self.data[idx:idx+self.block_size]
        y=self.data[idx+self.block_size]
        
        return x, y
  
  
BLOCK_SIZE = 8
BATCH_SIZE = 256


### 🔹 Dataset Object
Dataloader

**Purpose**: PyTorch Dataset & DataLoader Pipeline.

- Wraps tensors in iterable batches, handles multi-threaded worker loading, dynamic collation, and shuffling.


In [ ]:
train_dataset=TinyStoriesDataset(
    train_encoded, BLOCK_SIZE
)

valid_dataset=TinyStoriesDataset(
    valid_encoded, BLOCK_SIZE
)

train_loader=DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

valid_loader=DataLoader(
    valid_dataset,
    batch_size=BATCH_SIZE
)


### 🔹 MLP without BatchNorm

**Purpose**: Model Architecture Definition.

- Defines the network structure, layer projections, activations, and the forward propagation computation graph.


In [ ]:
class MLPWithoutBN(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size, block_size):
        super().__init__()
        
        self.embedding=nn.Embedding(
            vocab_size, embedding_dim
        )
        
        self.network=nn.Sequential(
            nn.Linear(
                embedding_dim*block_size, hidden_size
            ),
            nn.ReLU(),
            
            nn.Linear(
                hidden_size, vocab_size
            )
        )
        
    def forward(self, x):
        x=self.embedding(x)
        
        x=x.view(x.size(0), -1)
        
        return self.network(x)


### 🔹 MLP With BatchNorm

**Purpose**: Model Architecture Definition.

- Defines the network structure, layer projections, activations, and the forward propagation computation graph.


In [ ]:
class MLPWithBN(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size, block_size):
        super().__init__()
        
        self.embedding=nn.Embedding(
            vocab_size, embedding_dim
        )   
        
        self.network=nn.Sequential(
            nn.Linear(
                embedding_dim*block_size, 
                hidden_size
            ),
            nn.BatchNorm1d(hidden_size),
            nn.ReLU(),
            
            nn.Linear(
                hidden_size, vocab_size
            )
        ) 
        
    def forward(self, x):
        x=self.embedding(x)
        
        x=x.view(x.size(0), -1)
        
        return self.network(x)
    
    

EMBEDDING_DIM = 24
HIDDEN_SIZE = 256

EPOCHS = 10

LEARNING_RATE = 0.001


### 🔹 create model

**Purpose**: Core functional execution step.

- Executes the defined transformation, evaluation, or helper utility.


In [ ]:
model_without_bn = MLPWithoutBN(
    vocab_size=vocab_size,
    embedding_dim=EMBEDDING_DIM,
    hidden_size=HIDDEN_SIZE,
    block_size=BLOCK_SIZE
).to(device)

model_with_bn = MLPWithBN(
    vocab_size=vocab_size,
    embedding_dim=EMBEDDING_DIM,
    hidden_size=HIDDEN_SIZE,
    block_size=BLOCK_SIZE
).to(device)


### 🔹 Loss Function
Optimizers

**Purpose**: Loss Function & Optimizer Initialization.

- Configures optimization objective and update rule (e.g., Adam, SGD with momentum, weight decay).


In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer_without_bn = torch.optim.Adam(
    model_without_bn.parameters(),
    lr=LEARNING_RATE
)

optimizer_with_bn = torch.optim.Adam(
    model_with_bn.parameters(),
    lr=LEARNING_RATE
)


### 🔹 Evaluation Function

**Purpose**: Core functional execution step.

- Executes the defined transformation, evaluation, or helper utility.


In [ ]:
def evaluate_model(model, dataloader, criterion):
    model.eval()
    
    total_loss=0
    
    with torch.no_grad():
        for x, y in dataloader:
            x=x.to(device)
            y=y.to(device)
            
            logits=model(x)
            
            loss=criterion(logits, y)
            
            total_loss+=loss.item()
            
    return total_loss/len(dataloader)


### 🔹 Training Function

**Purpose**: Training & Optimization Loop.

- **Forward Pass**: Compute model predictions and loss.

- **Backward Pass**: `loss.backward()` calculates gradients via automatic differentiation.

- **Optimizer Step**: `optimizer.step()` updates trainable weights; `optimizer.zero_grad()` clears gradients.


In [ ]:
def train_model(model, train_loader, valid_loader, optimizer, criterion, epochs):
    train_losses=[]
    valid_losses=[]
    
    for epoch in range(epochs):
        model.train()
        
        running_loss=0
        
        for x, y in train_loader:
            x=x.to(device)
            y=y.to(device)
            
            optimizer.zero_grad()
            logits=model(x)
            
            loss=criterion(logits, y)
            
            loss.backward()
            optimizer.step()
            
            running_loss+=loss.item()
            
            
        train_loss=running_loss/len(train_loader)
        
        valid_loss=evaluate_model(
            model, valid_loader, criterion
        )
        
        train_losses.append(train_loss)
        valid_losses.append(valid_loss)
        
        print(
            f"Epoch {epoch+1}/{epochs}"
            f" | Train Loss: {train_loss:.4f}"
            f" | Validation Loss: {valid_loss:.4f}"
        )
        
    return train_losses, valid_losses


### 🔹 Train without BatchNorm

**Purpose**: Core functional execution step.

- Executes the defined transformation, evaluation, or helper utility.


In [ ]:
print("="*60)
print("Training WITHOUT Batch Normalization")

train_loss_without_bn, valid_loss_without_bn=train_model(
    model=model_without_bn,
    train_loader=train_loader,
    valid_loader=valid_loader,
    optimizer=optimizer_without_bn,
    criterion=criterion,
    epochs=EPOCHS
)


### 🔹 Train with BatchNorm

**Purpose**: Core functional execution step.

- Executes the defined transformation, evaluation, or helper utility.


In [ ]:
print("="*60)
print("Training WITH Batch Normalization")

train_loss_with_bn, valid_loss_with_bn=train_model(
    model=model_with_bn,
    train_loader=train_loader,
    valid_loader=valid_loader,
    optimizer=optimizer_with_bn,
    criterion=criterion,
    epochs=EPOCHS
)


### 🔹 Plot Loss Curves

**Purpose**: Evaluation, Metrics & Visualization.

- Evaluates model accuracy, F1-scores, loss convergence curves, and error distributions.


In [ ]:
plt.figure(figsize=(10,6))

plt.plot(
    train_loss_without_bn,
    label="Train (Without BN)"
)

plt.plot(
    valid_loss_without_bn,
    label="Validation (Without BN)"
)

plt.plot(
    train_loss_with_bn,
    label="Train (With BN)"
)

plt.plot(
    valid_loss_with_bn,
    label="Validation (With BN)"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Batch Normalization Comparison")
plt.legend()
plt.grid(True)

plt.show()


### 🔹 Text Generation Function

**Purpose**: Core functional execution step.

- Executes the defined transformation, evaluation, or helper utility.


In [ ]:
def generate_text(model, start_text, length=200):
    model.eval()
    
    context=[stoi[c] for c in start_text]
    
    for _ in range(length):
        x=torch.tensor(context[-BLOCK_SIZE:],
                       dtype=torch.long).unsqueeze(0).to(device)
        
        if x.shape[1]<BLOCK_SIZE:
            padding=torch.zeros(
                (1, BLOCK_SIZE-x.shape[1]),
                dtype=torch.long
            ).to(device)
            
            x=torch.cat([padding, x], dim=1)
            
        with torch.no_grad():
            logits=model(x)
            
            probs=torch.softmax(logits, dim=1)
            
            next_char=torch.multinomial(
                probs, num_samples=1
            ).item()
            
        context.append(next_char)
        
    return "".join(itos[i] for i in context)


### 🔹 Generate without BatchNorm

**Purpose**: Core functional execution step.

- Executes the defined transformation, evaluation, or helper utility.


In [ ]:
print("="*60)
print("WITHOUT BatchNorm")

print(generate_text(
    model_without_bn,
    "Once "
))


### 🔹 Generate with BatchNorm

**Purpose**: Core functional execution step.

- Executes the defined transformation, evaluation, or helper utility.


In [ ]:
print("="*60)
print("WITH BatchNorm")

print(generate_text(
    model_with_bn,
    "Once "
))


## 🎯 Summary & Key Takeaways
1. **Modular Execution**: Each component runs independently and validates intermediate tensor shapes and states.
2. **Core Insights**: Inspect the printed metrics, loss outputs, and visual distributions above.
3. **Next Lesson**: Applies these foundations to more advanced deep learning and transformer architectures.
